# Sound Manager AI - 추론 모듈 파인튜닝 (Colab T4)

런타임 → 런타임 유형 변경 → **T4 GPU**. 결과물 `smix-intent.gguf`를 플러그인의 `SoundManagerAI/models` 폴더에 넣으면 플러그인이 인터넷 없이 이 모델로 요청을 이해합니다.

In [ ]:
!git clone --depth 1 --branch claude/amazing-wozniak-hqy7y5 https://github.com/imhyensuk/sound_manager.git
%cd sound_manager/training/intent
!pip install -q "memopro[llm]" transformers peft datasets sentencepiece

## 1. 데이터 만들기
플러그인이 실제로 모델에 보내는 것과 같은 형식(시스템 프롬프트 + 세션 상태 + 요청 → JSON)으로 한국어/영어 예제를 생성합니다.

In [ ]:
!python make_dataset.py --out data --n 8000 --test 300
!head -c 600 data/test.jsonl

## 2. 학습 → 병합 → GGUF
memopro가 16비트 가중치를 파일에서 스트리밍하므로 T4에서도 3B 모델까지 학습할 수 있습니다 (`--base Qwen/Qwen2.5-3B-Instruct --budget 4GiB`).

In [ ]:
!python finetune_t4.py --base Qwen/Qwen2.5-1.5B-Instruct --data data --out out --budget 4GiB --epochs 2

## 3. 평가 (플러그인과 같은 문법·프롬프트로)
`smix_intent_eval`은 저장소를 빌드하면 생깁니다 (`cmake -S ../.. -B build -DSMIX_BUILD_PLUGIN=OFF && cmake --build build --target smix_intent_eval`).

In [ ]:
!cmake -S ../.. -B build -DSMIX_BUILD_PLUGIN=OFF -DSMIX_BUILD_TESTS=OFF -DSMIX_WITH_MEMOPRO=OFF > /dev/null && cmake --build build --target smix_intent_eval -j 4 > /dev/null
!./build/core/smix_intent_eval out/smix-intent.gguf data/test.jsonl 200

In [ ]:
from google.colab import files
files.download('out/smix-intent.gguf')